In [1]:
import pandas as pd
from pathlib import Path

file_path = Path("data") / "LAPD_NIBRS_Offenses_Dataset_20261004.csv"

# Load all rows while preserving leading zeros.
raw = pd.read_csv(file_path, dtype="string")

# Create a separate working copy.
nibrs = raw.copy()

print("Original rows:", raw.shape[0])
print("Original columns:", raw.shape[1])

raw.head()

Original rows: 489955
Original columns: 31


,CaseNo,UniqueNIBRNo,Date Rptd,Date OCC,Time OCC,AREA,AREA NAME,RPT Dist No,TotalOffenseCount,Group,...,Domestic Violence Crime,Hate Crime,Gang-related Crime,Transit-related Crime,Homeless-Victim Crime,Homeless-Suspect Crime,Homeless-Arrestee Crime,LOCATION,LAT,LON
0,25088246,25088246_13A_0,2025 May 15 02:23:02 PM,2025 May 15 02:21:00 PM,1421,05,Harbor,0558,3,A,...,No,No,Yes,No,No,No,No,400 N Palos Verdes St,33.746,-118.2817
1,25088246,25088246_290_2,2025 May 15 02:23:02 PM,2025 May 15 02:21:00 PM,1421,05,Harbor,0558,3,A,...,No,No,Yes,No,No,No,No,400 N Palos Verdes St,33.746,-118.2817
2,26170796,26170796_13B_0,2026 Aug 31 03:47:10 AM,2026 Aug 31 02:00:00 AM,0200,01,Central,0166,1,A,...,No,No,No,No,No,No,No,600 S San Pedro St,34.0423,-118.2453
3,26170797,26170797_13B_0,2026 Aug 31 02:55:11 AM,2026 Aug 31 01:30:21 AM,0130,15,N Hollywood,1503,1,A,...,No,No,No,No,No,No,No,12300 Sherman Way,34.2013,-118.4011
4,26170801,26170801_13B_0,2026 Aug 30 11:43:17 PM,2026 Aug 30 11:30:00 AM,1130,03,Southwest,0398,1,A,...,Yes,No,No,No,No,No,No,4100 S Hoover St,34.0094,-118.2871


Inspecting the original data beforehand.

In [2]:
# Check the loaded column types and non-missing counts.
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 489955 entries, 0 to 489954
Data columns (total 31 columns):
 #   Column                   Non-Null Count   Dtype 
---  ------                   --------------   ----- 
 0   CaseNo                   489955 non-null  string
 1   UniqueNIBRNo             489955 non-null  string
 2   Date Rptd                482168 non-null  string
 3   Date OCC                 489955 non-null  string
 4   Time OCC                 489955 non-null  string
 5   AREA                     489955 non-null  string
 6   AREA NAME                489955 non-null  string
 7   RPT Dist No              489955 non-null  string
 8   TotalOffenseCount        489955 non-null  string
 9   Group                    474475 non-null  string
 10  NIBR Code                488806 non-null  string
 11  NIBR Description         489955 non-null  string
 12  Crime Against            487658 non-null  string
 13  Premise Cd               467102 non-null  string
 14  Premise Desc             489846

### Initial inspection: dataset structure and data types

The dataset contains 489,955 rows and 31 columns. Each row represents
a reported NIBRS offense. Multiple offenses may belong to the same case.

All columns were intentionally loaded as strings to preserve the original
text formatting, including leading zeros in identifiers and occurrence times.
This does not mean that every column should remain a string after cleaning.

The Non-Null Count shows how many values are present in each column.
Some fields are incomplete. For example:
- Date Rptd has 7,787 missing values.
- Premise Desc has 109 missing values.
- LAT and LON each have 131 missing values.

These counts do not yet tell us whether missing values occur in the same
records or whether populated values are valid. Additional checks are needed
for blank strings, invalid values, duplicates, and inconsistent formatting.

### Implications for cleaning

- Convert Date OCC and Date Rptd to datetime values and count conversion failures.
- Convert LAT and LON to numeric values and check coordinate validity.
- Convert TotalOffenseCount and TotalVictimCount to numeric values before
  validating or using them in calculations.
- Keep identifiers and category codes as strings to preserve leading zeros.
- Keep Time OCC as a string and validate its HHMM structure separately.
- Investigate missing values by column before deciding how to handle them.
- Preserve the original data and record the effects of each cleaning step.

No rows have been removed or values changed during this inspection.

### Check missing values and duplicates

This step counts missing values in each column and checks for exact duplicate
rows and repeated offense IDs.

UniqueNIBRNo should identify an individual offense. CaseNo may repeat because
one case can contain multiple offenses. Repeated case numbers alone are not
a reason to remove rows.

These checks inspect the original data without changing it.

In [3]:
# Count missing values for every column.
missing_before = raw.isna().sum().sort_values(ascending=False)
display(missing_before)

# Check duplicates and identifiers.
print("Exact duplicate rows:", raw.duplicated().sum())
print("Missing offense IDs:", raw["UniqueNIBRNo"].isna().sum())
print("Repeated offense IDs beyond the first:",
      raw["UniqueNIBRNo"].duplicated().sum())

print("Unique offenses:", raw["UniqueNIBRNo"].nunique())
print("Unique cases:", raw["CaseNo"].nunique())

Weapon Used Cd             345971
Weapon Desc                345971
Premise Cd                  22853
Group                       15480
Date Rptd                    7787
Crime Against                2297
NIBR Code                    1149
Status                       1044
Status Desc                  1042
LAT                           131
LON                           131
Premise Desc                  109
AREA                            0
AREA NAME                       0
CaseNo                          0
NIBR Description                0
RPT Dist No                     0
TotalOffenseCount               0
UniqueNIBRNo                    0
Date OCC                        0
Time OCC                        0
TotalVictimCount                0
Victim Shot                     0
Hate Crime                      0
Domestic Violence Crime         0
Gang-related Crime              0
Transit-related Crime           0
Homeless-Suspect Crime          0
Homeless-Victim Crime           0
LOCATION      

Exact duplicate rows: 0
Missing offense IDs: 0
Repeated offense IDs beyond the first: 0
Unique offenses: 489955
Unique cases: 440121


### Findings: missing values and duplicates

**Duplicates and identifiers**
- There are 0 exact duplicate rows.
- There are 0 missing offense IDs and 0 repeated offense IDs.
- All 489,955 rows have a distinct UniqueNIBRNo.
- There are 440,121 distinct CaseNo values. Multiple offenses can belong
  to one case, so repeated case numbers will be retained.
- No duplicate removal is needed based on these checks.

**Missing values**
- Weapon Used Cd and Weapon Desc each have 345,971 missing values.
  Missing weapon information will not be interpreted automatically as
  indicating that no weapon was involved.
- Date OCC and Time OCC have no missing values, but their validity still
  needs to be checked.
- Date Rptd has 7,787 missing values. Missing report dates alone will not
  exclude records because the project uses occurrence dates.
- LAT and LON each have 131 missing values. We still need to check whether
  these are the same records. Missing coordinates affect mapping but may
  not prevent a record from contributing to reporting-district summaries.
- Other missing category values will be reviewed individually.

**Proposed handling**
Avoid dropping every row containing a missing value. Preserve missing
values unless there is a justified rule for filling them. Determine which
records are eligible for each analysis based on the fields it requires.

These results describe the full original dataset. Missing-value counts
will also be checked after restricting the analysis to 2025 onward.

In [4]:
print(missing_before.to_string())

Weapon Used Cd             345971
Weapon Desc                345971
Premise Cd                  22853
Group                       15480
Date Rptd                    7787
Crime Against                2297
NIBR Code                    1149
Status                       1044
Status Desc                  1042
LAT                           131
LON                           131
Premise Desc                  109
AREA                            0
AREA NAME                       0
CaseNo                          0
NIBR Description                0
RPT Dist No                     0
TotalOffenseCount               0
UniqueNIBRNo                    0
Date OCC                        0
Time OCC                        0
TotalVictimCount                0
Victim Shot                     0
Hate Crime                      0
Domestic Violence Crime         0
Gang-related Crime              0
Transit-related Crime           0
Homeless-Suspect Crime          0
Homeless-Victim Crime           0
LOCATION      

---
## Part 2: Cleaning with shared logic (`clean_nibrs.py`)

The cells above are my original inspection. They are unchanged.

From here on, the notebook calls functions from **`clean_nibrs.py`**, the same
file the command-line run uses (`python clean_nibrs.py`). That keeps the script
and the notebook consistent. The rules and the reasons for them are in
**`plan.md`**. Each step below has:

- a Markdown cell explaining **what** the step does and **why**,
- a code cell that runs the step and prints its checks,
- a Markdown cell recording **what changed** and the result of the check.

Cleaning keeps all 489,955 rows. Analysis scope (occurrence on or after
2025-01-01) and eligibility for each metric are recorded as columns, so every
excluded record can be accounted for.

### Correction to my earlier loading note

My earlier note said the columns were loaded as strings "to preserve the original
text formatting." More precisely:

- `dtype="string"` keeps **leading zeros** (for example `AREA` `05`,
  `RPT Dist No` `0558`, `Time OCC` `0000`) and stops pandas from guessing types.
- `read_csv` **still applies its default missing-value tokens**. Empty fields
  and strings such as `NA`, `NULL` or `N/A` become missing (`<NA>`). So the
  load does not keep every original text value literally.

The next cell checks which raw tokens actually became missing. It reloads the
file with NA detection turned off and compares the two loads. It also confirms
that the copy loaded above (`raw`) matches the script's loader, and that the
original CSV's SHA-256 matches the recorded fingerprint of our fixed dataset.

In [5]:
import clean_nibrs as cn

sha_before = cn.sha256_file(file_path)
print("SHA-256 of original CSV:", sha_before)
print("Matches recorded fingerprint:", sha_before == cn.EXPECTED_SHA256)
assert sha_before == cn.EXPECTED_SHA256, "The CSV is not the fixed project dataset"

# The notebook's raw copy (cell 1) must equal the script's loader.
assert raw.equals(cn.load_raw(file_path))
print("Notebook `raw` is identical to clean_nibrs.load_raw():", True)

tokens = cn.check_missing_tokens(file_path, raw)
display(pd.DataFrame(
    [(col, tok, n) for col, found in tokens["tokens_converted_to_na"].items() for tok, n in found.items()],
    columns=["column", "raw token that became NA", "rows"],
))
print("Only empty fields became NA:", tokens["only_empty_fields_became_na"])
print("All non-missing values identical to the literal text:",
      tokens["non_missing_values_identical_to_literal_text"])

SHA-256 of original CSV: 97a8dbadafe60eee4b4d86e99769df3aaa23307d45539283879414f8760b39e5
Matches recorded fingerprint: True
Notebook `raw` is identical to clean_nibrs.load_raw(): True


,column,raw token that became NA,rows
0,Date Rptd,<empty field>,7787
1,Group,<empty field>,15480
2,NIBR Code,<empty field>,1149
3,Crime Against,<empty field>,2297
4,Premise Cd,<empty field>,22853
5,Premise Desc,<empty field>,109
6,Weapon Used Cd,<empty field>,345971
7,Weapon Desc,<empty field>,345971
8,Status,<empty field>,1044
9,Status Desc,<empty field>,1042


Only empty fields became NA: True
All non-missing values identical to the literal text: True


**Result:** in this file, every value that became missing was an **empty field**.
No literal `NA`/`NULL`-style text was converted, and all non-missing values match
the literal text exactly. That is a finding about this particular file, not a
general property of `dtype="string"`.

### Record checks: expected results written *before* any transformation

These five records were chosen during planning because they are difficult cases.
Their expected results (plan.md, section 8) are fixed in `cn.RECORD_CHECKS`
before any cleaning step runs. The table below shows each record's
**original raw values**. The actual results are compared near the end of the
notebook.

| # | Record | Why it is difficult |
|---|---|---|
| R1 | `25088246_290_2` | One of 3 offense rows in a case; case-level totals repeated; daytime (14:21) |
| R2 | `26172245_220_0` | `Time OCC` 0000; month 2026-08; weapon missing |
| R3 | `C259005916_23H_0` | `RPT Dist No` holds only the area code `10`; coordinates (0, 0) |
| R4 | `25000271_13B_0` | Occurred 2024-12-31 23:55 (night), reported 2025-01-01: outside scope by occurrence date |
| R5 | `26173656_90D_0` | Coordinates about 32 km from its district's typical location; partial month; 06:45 (not night) |

In [6]:
record_ids = [check["uid"] for check in cn.RECORD_CHECKS]
original_records = raw.set_index("UniqueNIBRNo").loc[record_ids, [
    "CaseNo", "Date Rptd", "Date OCC", "Time OCC", "AREA", "AREA NAME", "RPT Dist No",
    "TotalOffenseCount", "TotalVictimCount", "Weapon Used Cd", "LOCATION", "LAT", "LON",
]]
display(original_records)

expected_table = pd.DataFrame(
    [(c["id"], field, value) for c in cn.RECORD_CHECKS for field, value in c["expected"].items()],
    columns=["record", "field", "expected"],
)
display(expected_table)

,CaseNo,Date Rptd,Date OCC,Time OCC,AREA,AREA NAME,RPT Dist No,TotalOffenseCount,TotalVictimCount,Weapon Used Cd,LOCATION,LAT,LON
UniqueNIBRNo,,,,,,,,,,,,,
25088246_290_2,25088246,2025 May 15 02:23:02 PM,2025 May 15 02:21:00 PM,1421,05,Harbor,0558,3,5,02,400 N Palos Verdes St,33.746,-118.2817
26172245_220_0,26172245,2026 Sep 01 08:22:37 PM,2026 Aug 31 12:00:00 AM,0000,15,N Hollywood,1555,1,1,<NA>,10900 Hortense St,34.1545,-118.3683
C259005916_23H_0,C259005916,2025 Feb 10 06:51:23 PM,2025 Feb 10 03:00:00 PM,1500,10,West Valley,10,1,1,<NA>,Victory Blvd & Tampa Ave,0,0
25000271_13B_0,25000271,2025 Jan 01 09:06:34 AM,2024 Dec 31 11:55:00 PM,2355,06,Hollywood,0647,1,1,08,6200 Sunset Blvd,34.098,-118.3237
26173656_90D_0,26173656,2026 Sep 03 07:16:01 PM,2026 Sep 03 06:45:00 AM,0645,01,Central,0138,1,0,<NA>,Alameda & 2nd,34.1244,-117.9063


,record,field,expected
0,R1,in_scope,True
1,R1,occ_year_month,2025-05
2,R1,RPT Dist No,0558
3,R1,AREA NAME,Harbor
4,R1,TotalOffenseCount,3
5,R1,TotalVictimCount,5
6,R1,case_rows_in_file,3
7,R1,toc_mismatch,False
8,R1,Gang-related Crime,True
9,R1,map_eligible,True


### Step 1: Standardize text

**What:**
- Strip the spaces padding `AREA NAME` (every value was padded to 20 characters).
- Collapse repeated internal spaces in `NIBR Description` (for example
  `GTA  - 240` → `GTA - 240`).

**What is not changed:** upper/lower case, category wording, codes, and IDs.
Legacy upper-case descriptions stay distinct, and leading zeros are untouched.

**Check:**
- count the cells that changed;
- compare distinct values before and after;
- list the descriptions that merged because of the change;
- confirm the length of every code column is unchanged.

In [7]:
nibrs, text_report = cn.standardize_text(nibrs)

for column in ["AREA NAME", "NIBR Description"]:
    r = text_report[column]
    print(f"{column}: {r['changed_cells']:,} cells changed; distinct values "
          f"{r['distinct_before']:,} -> {r['distinct_after']:,}")

display(pd.DataFrame(text_report["AREA NAME"]["examples"]).assign(
    before=lambda d: d["before"].map(repr), after=lambda d: d["after"].map(repr)))
print("NIBR Description values merged by whitespace cleanup:")
for after, befores in text_report["NIBR Description"]["merged_values"].items():
    print(f"  {after!r} <- {[repr(b) for b in befores]}")
print("Leading-zero columns keep their lengths:", text_report["leading_zero_lengths_unchanged"])
print("Rows:", len(nibrs))

AREA NAME: 489,955 cells changed; distinct values 21 -> 21
NIBR Description: 82,058 cells changed; distinct values 2,409 -> 2,398


,before,after
0,'Harbor ','Harbor'
1,'Central ','Central'
2,'N Hollywood ','N Hollywood'
3,'Southwest ','Southwest'
4,'Hollywood ','Hollywood'


NIBR Description values merged by whitespace cleanup:
  '210 - ROBBERY' <- ["'210  - ROBBERY'", "'210 - ROBBERY'"]
  '230 - ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT' <- ["'230  - ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT'", "'230 - ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT'"]
  '236 - PC - F - False Imprisonment - 100' <- ["'236 - PC - F - False Imprisonment  - 100'", "'236 - PC - F - False Imprisonment - 100'"]
  '278 - PC - F - Child Stealing - 100' <- ["'278 - PC - F - Child Stealing  - 100'", "'278 - PC - F - Child Stealing - 100'"]
  '286(B)(2) - PC - F - Sodomy W/Person Under 16 Years - 11A' <- ["'286(B)(2) - PC - F - Sodomy W/Person Under 16 Years  - 11A'", "'286(B)(2) - PC - F - Sodomy W/Person Under 16 Years - 11A'"]
  '810 - SEX,UNLAWFUL(INC MUTUAL CONSENT, PENETRATION W/ FRGN OBJ' <- ["'810  - SEX,UNLAWFUL(INC MUTUAL CONSENT, PENETRATION W/ FRGN OBJ'", "'810 - SEX,UNLAWFUL(INC MUTUAL CONSENT, PENETRATION W/ FRGN OBJ'"]
  '812 - CRM AGNST CHLD (13 OR UNDER) (1

**What changed:**

| Column | Cells changed | Distinct values |
|---|---|---|
| `AREA NAME` | 489,955 | 21 → 21 (only the padding was removed) |
| `NIBR Description` | 82,058 | 2,409 → 2,398 |

The 11 merges in `NIBR Description` are pairs that differed **only** by doubled
spaces. Lengths of `AREA`, `RPT Dist No`, `Premise Cd`, `Weapon Used Cd` and
`Time OCC` are unchanged, so leading zeros are intact. The row count is
unchanged.

### Step 2: Convert types

**What:**

| Column(s) | Converted to | Note |
|---|---|---|
| `Date OCC`, `Date Rptd` | datetime | Explicit format `%Y %b %d %I:%M:%S %p`, naive local time |
| `TotalOffenseCount`, `TotalVictimCount` | whole numbers (`Int64`) | |
| `LAT`, `LON` | decimal numbers (`Float64`) | |
| 8 Yes/No columns | `True`/`False` | |

Identifiers and codes (`CaseNo`, `UniqueNIBRNo`, `AREA`, `RPT Dist No`,
`NIBR Code`, `Premise Cd`, `Weapon Used Cd`, `Status`) and `Time OCC` stay as
text, to keep their leading zeros.

**Check:**
- count failed conversions (a value present before but missing after);
- confirm the Yes/No columns contained only `Yes`/`No`, with `True` counts
  equal to the original `Yes` counts.

In [8]:
nibrs, type_report = cn.convert_types(nibrs)

display(pd.Series(type_report["failed_conversions"], name="failed conversions").to_frame())
display(pd.DataFrame(type_report["yes_no"]).T)
display(nibrs[["Date Rptd", "Date OCC", "TotalOffenseCount", "TotalVictimCount", "LAT", "LON",
               "Victim Shot", "AREA", "RPT Dist No", "Time OCC"]].dtypes.to_frame("dtype"))

,failed conversions
Date Rptd,0
Date OCC,0
TotalOffenseCount,0
TotalVictimCount,0
LAT,0
LON,0
Victim Shot,0
Domestic Violence Crime,0
Hate Crime,0
Gang-related Crime,0


,yes_before,true_after,unexpected_values
Victim Shot,2062,2062,[]
Domestic Violence Crime,46396,46396,[]
Hate Crime,1800,1800,[]
Gang-related Crime,6932,6932,[]
Transit-related Crime,15521,15521,[]
Homeless-Victim Crime,13890,13890,[]
Homeless-Suspect Crime,20267,20267,[]
Homeless-Arrestee Crime,31363,31363,[]


,dtype
Date Rptd,datetime64[us]
Date OCC,datetime64[us]
TotalOffenseCount,Int64
TotalVictimCount,Int64
LAT,Float64
LON,Float64
Victim Shot,boolean
AREA,string
RPT Dist No,string
Time OCC,string


**What changed:** 14 columns now have proper types, with **0 failed conversions**.
The missing values that already existed stay missing: for example, 7,787 empty
`Date Rptd` values remain `NaT`. They are not filled. Every Yes/No column
contained only `Yes`/`No`.

### Step 3: Quality flags (nothing is changed or removed)

**What:** new flag columns mark records that need care. The original values are
kept. The most important flags:

| Flag | Meaning |
|---|---|
| `rpt_dist_area_only` | `RPT Dist No` holds a 2-digit area code instead of a 4-digit district. No district is guessed. |
| `district_eligible` | Has a 4-digit district, so the record can enter district summaries |
| `toc_mismatch` | A row's `TotalOffenseCount` differs from the number of rows for its `CaseNo` in the file |
| `tvc_conflict` | `TotalVictimCount` differs between rows of the same case |
| `nibr_code_missing`, `nibr_desc_code_conflict` | Offense-code problems. These rows **still count** as offenses. |
| `same_code_repeat_in_case` | The same NIBR code appears more than once in a case. These rows are kept, because each has its own `UniqueNIBRNo`. |
| `location_suspected_generalized` | `LOCATION` equals `RPT Dist No`, suggesting a district-level point (not verified) |
| `rpt_before_occ`, `occ_implausible_year`, `case_field_inconsistent` | Date and case consistency checks |

**Why `TotalOffenseCount` and `TotalVictimCount` are never summed:** they are
case-level values repeated on every offense row. Adding them up across rows
double-counts, as the next cell shows for victims.

In [9]:
nibrs = cn.add_quality_flags(nibrs)

flag_cols = ["rpt_dist_area_only", "district_eligible", "toc_mismatch", "tvc_conflict",
             "nibr_code_missing", "nibr_desc_code_conflict", "same_code_repeat_in_case",
             "location_suspected_generalized", "rpt_before_occ", "occ_implausible_year",
             "case_field_inconsistent"]
display(nibrs[flag_cols].sum().to_frame("rows flagged (all years)"))
print("toc_mismatch: rows =", int(nibrs["toc_mismatch"].sum()),
      "| cases =", nibrs.loc[nibrs["toc_mismatch"], "CaseNo"].nunique())
print("tvc_conflict: cases =", nibrs.loc[nibrs["tvc_conflict"], "CaseNo"].nunique())

# Why case-level totals are not summed: example case 25088246 (3 offense rows).
display(nibrs.loc[nibrs["CaseNo"] == "25088246",
                  ["UniqueNIBRNo", "NIBR Code", "TotalOffenseCount", "TotalVictimCount", "case_rows_in_file"]])
print("Summing TotalVictimCount over these 3 rows would give",
      int(nibrs.loc[nibrs["CaseNo"] == "25088246", "TotalVictimCount"].sum()),
      "but the case reports 5 victims once per row.")

,rows flagged (all years)
rpt_dist_area_only,558
district_eligible,489397
toc_mismatch,96
tvc_conflict,39
nibr_code_missing,1149
nibr_desc_code_conflict,16
same_code_repeat_in_case,10425
location_suspected_generalized,10412
rpt_before_occ,523
occ_implausible_year,3


toc_mismatch: rows = 96 | cases = 80
tvc_conflict: cases = 17


,UniqueNIBRNo,NIBR Code,TotalOffenseCount,TotalVictimCount,case_rows_in_file
0,25088246_13A_0,13A,3,5,3
1,25088246_290_2,290,3,5,3
11526,25088246_23F_1,23F,3,5,3


Summing TotalVictimCount over these 3 rows would give 15 but the case reports 5 victims once per row.


**What changed:** only new flag columns were added. No values changed.

- `toc_mismatch` flags **96 rows in 80 cases**. That is the row-level rule from
  plan.md section 1.4; my first draft's first-row method had reported 55 cases.
- `tvc_conflict` flags 17 cases. Because some cases conflict, **no victim total
  is produced**. Picking one of the conflicting values would be arbitrary.
- 558 rows have an area-only `RPT Dist No`, and 10,412 rows have a suspected
  generalized location.

### Step 4: Time fields and nighttime

**What:**
- `occ_date` and `occ_year_month` (`YYYY-MM`) are derived from the validated
  `Date OCC`. Years stay separate, so `2025-01` and `2026-01` are different months.
- `occ_minute_of_day` is derived from `Time OCC`.
- `time_is_0000` and `time_is_1200` flag the two heaped values. 0000 is **not**
  set to missing.
- **`night` = 20:00 inclusive to 06:00 exclusive.** This is a *fixed-clock
  approximation* of nighttime. It ignores seasonal sunset and sunrise, and clock
  time does not measure lighting. 0000 falls inside the window, so nighttime
  counts are reported both including and excluding 0000.

**Check:**
- every `Time OCC` is a valid `HHMM`;
- `Time OCC` equals the time inside `Date OCC` for every row;
- minute of day falls within 0–1439;
- the night-window boundaries behave as intended (19:59 no, 20:00 yes,
  05:59 yes, 06:00 no).

In [10]:
nibrs, time_report = cn.add_time_fields(nibrs)
print(time_report)

boundary = (nibrs.drop_duplicates("Time OCC")
                 .loc[lambda d: d["Time OCC"].isin(["1959", "2000", "0000", "0559", "0600", "1200"]),
                      ["Time OCC", "occ_minute_of_day", "night"]]
                 .sort_values("occ_minute_of_day"))
display(boundary)
print("Time OCC 0000 rows (all years):", int(nibrs["time_is_0000"].sum()))
print("Time OCC 1200 rows (all years):", int(nibrs["time_is_1200"].sum()))

{'invalid_hhmm': 0, 'time_matches_date_occ': 489955, 'minute_min': 0, 'minute_max': 1439}


,Time OCC,occ_minute_of_day,night
53,0000,0,True
11591,0559,359,True
90,0600,360,False
43,1200,720,False
6966,1959,1199,False
9,2000,1200,True


Time OCC 0000 rows (all years): 15333
Time OCC 1200 rows (all years): 13314


**What changed:** new derived columns only.

- `Time OCC` is valid `HHMM` in every row.
- It equals the time inside `Date OCC` in **all 489,955 rows**, so `time_conflict`
  flags 0 rows.
- The boundary table confirms 19:59 and 06:00 are not night, while 20:00, 00:00
  and 05:59 are.
- 0000 appears in 15,333 rows across all years. These values are heaped, but the
  source does not say why, so no meaning is assumed.

### Step 5: Coordinate checks (separate flags; nothing moved or replaced)

**What:**

| Flag | Meaning | Effect |
|---|---|---|
| `coord_missing` | `LAT`/`LON` empty in the source | Not mappable |
| `coord_nonnumeric`, `coord_nonfinite`, `coord_out_of_global_bounds` | Basic validity: numeric, finite, latitude within ±90 and longitude within ±180 | Not mappable |
| `coord_zero` | Exactly (0, 0) | Set to missing (0, 0 is not a location in Los Angeles). No replacement. |
| **`map_eligible`** | Passes all the checks above | May be placed on a map. *This does not establish that the location is accurate.* |
| `coord_outside_basic_range` | Outside a rough box around the City of LA | Review flag only |
| `coord_far_from_district_review` | More than 5 km from the median point of its district's street-address rows | Review flag only. Unusual does not mean wrong. |

**No coordinate flag removes a record from district counts.** District
summaries use `RPT Dist No`, not coordinates.

**Reading `coord_far_from_district_review`:** `True` means flagged for review.
`False` means **"not flagged"**, not "verified close". The distance is computed
only for map-eligible rows with a 4-digit district. For other rows
(`dist_km_from_district_median` missing), the row was **not assessed**, and
`False` does not establish that its coordinates are close to the district
median. No median or distance is filled in for those rows.

In [11]:
nibrs = cn.add_coordinate_flags(nibrs)
coord_flags = ["coord_missing", "coord_nonnumeric", "coord_nonfinite", "coord_out_of_global_bounds",
               "coord_zero", "map_eligible", "coord_outside_basic_range", "coord_far_from_district_review",
               "location_suspected_generalized"]
display(nibrs[coord_flags].sum().to_frame("rows (all years)"))
print("LAT missing after cleaning:", int(nibrs["LAT"].isna().sum()), "(131 empty + 550 zero)")
display(nibrs.loc[nibrs["UniqueNIBRNo"] == "26173656_90D_0",
                  ["UniqueNIBRNo", "AREA NAME", "RPT Dist No", "LOCATION", "LAT", "LON",
                   "dist_km_from_district_median", "coord_outside_basic_range",
                   "coord_far_from_district_review", "map_eligible"]])

,rows (all years)
coord_missing,131
coord_nonnumeric,0
coord_nonfinite,0
coord_out_of_global_bounds,0
coord_zero,550
map_eligible,489274
coord_outside_basic_range,161
coord_far_from_district_review,2479
location_suspected_generalized,10412


LAT missing after cleaning: 681 (131 empty + 550 zero)


,UniqueNIBRNo,AREA NAME,RPT Dist No,LOCATION,LAT,LON,dist_km_from_district_median,coord_outside_basic_range,coord_far_from_district_review,map_eligible
106,26173656_90D_0,Central,0138,Alameda & 2nd,34.1244,-117.9063,32.029,True,True,True


**What changed:** 550 (0, 0) pairs became missing, so `LAT`/`LON` missing goes
from 131 to 681. Nothing else in `LAT`/`LON` changed.

- 0 values are non-numeric, non-finite, or outside global bounds.
- Across all years, 161 points fall outside the rough LA box and 2,479 are more
  than 5 km from their district's median. Both are **review flags only**.
- 681 rows have no distance, so they were **not assessed** for the distance
  flag. They are the 558 area-only `RPT Dist No` rows and 123 rows with a
  4-digit district but non-mappable coordinates. Their `False` flag means
  "not flagged", not "close to the district". Step 7 reconciles these groups.
- Record R5 keeps its original coordinates and is flagged for review. It is
  still map-eligible, and it is eligible for an **all-hours** summary of district
  0138. Its time (06:45) is outside the night window, so it contributes
  **nothing** to nighttime counts or to `district_month_night.csv`.

### Step 6: Analysis scope and accounting for every excluded record

**What:** the scope is `Date OCC >= 2025-01-01`, using the **occurrence** date,
not the report date. Out-of-scope rows are **kept** in the cleaned file, with a
reason code:

| `scope_reason` | Rule |
|---|---|
| `pre_2024` | Occurred before 2024 (includes 3 dates before 1900) |
| `2024_pre_rms` | 2024-01-01 to 2024-03-06, before LAPD's NIBRS-aligned records system started |
| `2024_rms_transition` | 2024-03-07 to 2024-12-31 |
| `in_scope` | On or after 2025-01-01 |

`calendar_coverage` labels each in-scope month:
- `complete`: the whole calendar month is inside the data period.
- `partial`: 2026-09 only, because the last occurrence date in the file is
  2026-09-19.

All September records are kept. **Calendar coverage is not reporting
completeness:** any month can still receive late reports.

In [12]:
nibrs = cn.apply_scope(nibrs)
in_scope = nibrs["in_scope"]

scope_table = nibrs["scope_reason"].value_counts().reindex(
    ["pre_2024", "2024_pre_rms", "2024_rms_transition", "in_scope"]).to_frame("rows")
scope_table.loc["TOTAL"] = scope_table["rows"].sum()
display(scope_table)
print("In-scope distinct cases:", nibrs.loc[in_scope, "CaseNo"].nunique())

display(pd.DataFrame({
    "missing (all rows)": nibrs[raw.columns].isna().sum(),
    "missing (in scope)": nibrs.loc[in_scope, raw.columns].isna().sum(),
}).loc[lambda d: d.iloc[:, 0] > 0])

night = in_scope & nibrs["night"]
zero = nibrs["time_is_0000"]
print("In-scope Time OCC 0000:", int((in_scope & zero).sum()))
print("In-scope night offenses including 0000:", int(night.sum()))
print("In-scope night offenses excluding 0000:", int((night & ~zero).sum()))

,rows
scope_reason,
pre_2024,6824
2024_pre_rms,1351
2024_rms_transition,101689
in_scope,380091
TOTAL,489955


In-scope distinct cases: 344186


,missing (all rows),missing (in scope)
Date Rptd,7787,6577
Group,15480,10153
NIBR Code,1149,72
Crime Against,2297,203
Premise Cd,22853,10959
Premise Desc,109,4
Weapon Used Cd,345971,277461
Weapon Desc,345971,277461
Status,1044,36
Status Desc,1042,36


In-scope Time OCC 0000: 10330
In-scope night offenses including 0000: 137415
In-scope night offenses excluding 0000: 127085


**What changed:** no rows were removed. The scope accounting is:

| Reason | Rows |
|---|---|
| `pre_2024` | 6,824 |
| `2024_pre_rms` | 1,351 |
| `2024_rms_transition` | 101,689 |
| **In scope** | **380,091** (344,186 distinct cases) |
| **Total** | **489,955** |

In scope, `Time OCC` is 0000 in 10,330 rows. That total was computed **before**
any exclusion. Night offenses: **137,415 including 0000**, **127,085 excluding**.

### Step 7: Monthly nighttime counts by reporting district

**Which records count:** in-scope **and** `night` **and** `district_eligible`.
Daytime records (such as R1 and R5) contribute to no nighttime count, even
when they are otherwise eligible for district summaries.

**Columns of `district_month_night.csv`:**
- Offense counts (rows): `night_offenses_incl_0000`, `night_offenses_excl_0000`
  and `night_offenses_0000`.
- **Distinct cases**: distinct `CaseNo` among the same rows. These are *not*
  assumed to be distinct incidents.
- `calendar_coverage`.
- `days_since_month_end_to_last_report`: days from the month's end to the
  latest report date in the file (2026-09-28). It is blank for the partial month.

The table contains **observed** district-month combinations only. A combination
with no records is absent, not zero. The complete district × month grid and any
persistence threshold are deferred.

**Missing grouping keys:** records with an area-only `RPT Dist No` are left out
of district rows. They are reported separately as "unassigned" by area and month.
`occ_year_month` and `AREA` are never missing.

**Internal joins (no external dataset is merged):** month context
(`calendar_coverage`, `days_since_month_end_to_last_report`) is attached with
**many-to-one left joins on `occ_year_month`**:
- to the district-month table,
- to the unassigned table,
- to the monthly summary.

Each join is audited for:
- left, right and result row counts (the result must equal the left rows, so
  no extra rows);
- duplicate keys on the right side;
- unmatched left keys;
- right keys not used.

Two lookups also act as joins and are audited: the per-`CaseNo` counts used for
case flags, and each district's median point used for the distance review flag.

`cn.finalize` also builds the sample, compares the record checks, recounts 3
cells directly from the raw text, and runs the automated validation.

In [13]:
results = cn.finalize(file_path, raw, sha_before, tokens, text_report, type_report, time_report, nibrs)
district_month = results["district_month"]
unassigned_month = results["unassigned_month"]

print("Observed district-month rows:", len(district_month),
      "| districts:", district_month["RPT Dist No"].nunique())
display(district_month.head(8))

print("Reconciliation (including 0000):",
      int(district_month["night_offenses_incl_0000"].sum()), "+",
      int(unassigned_month["night_offenses_incl_0000"].sum()), "=",
      int(district_month["night_offenses_incl_0000"].sum() + unassigned_month["night_offenses_incl_0000"].sum()),
      "vs in-scope night", int(night.sum()))
print("Reconciliation (excluding 0000):",
      int(district_month["night_offenses_excl_0000"].sum()), "+",
      int(unassigned_month["night_offenses_excl_0000"].sum()), "=",
      int(district_month["night_offenses_excl_0000"].sum() + unassigned_month["night_offenses_excl_0000"].sum()),
      "vs", int((night & ~zero).sum()))

display(results["month_summary"])
display(pd.DataFrame(results["recount"]))

# Audits of the internal joins and lookups.
display(pd.DataFrame(results["join_audits"])[
    ["join", "key", "left_rows", "right_rows", "result_rows", "extra_rows",
     "right_duplicate_keys", "unmatched_left_rows", "right_keys_not_in_left"]])
display(pd.DataFrame(results["lookup_audits"])[
    ["join", "key", "left_rows", "right_rows", "result_rows", "extra_rows", "unmatched_left_rows"]])

# District-median lookup: rows looked up vs. rows not looked up (distance stays missing).
median_audit = results["lookup_audits"][1]
display(pd.DataFrame(median_audit["not_looked_up"]).T)
display(pd.Series(median_audit["reconciliation"], name="rows").to_frame())

Observed district-month rows: 21290 | districts: 1140


,RPT Dist No,AREA,AREA NAME,occ_year_month,calendar_coverage,days_since_month_end_to_last_report,night_offenses_incl_0000,night_offenses_excl_0000,night_offenses_0000,night_distinct_cases_incl_0000,night_distinct_cases_excl_0000
0,0101,01,Central,2025-01,complete,605,10,10,0,9,9
1,0101,01,Central,2025-02,complete,577,10,9,1,9,8
2,0101,01,Central,2025-03,complete,546,11,10,1,9,8
3,0101,01,Central,2025-04,complete,516,16,14,2,13,12
4,0101,01,Central,2025-05,complete,485,12,10,2,12,10
5,0101,01,Central,2025-06,complete,455,8,5,3,7,5
6,0101,01,Central,2025-07,complete,424,17,15,2,15,14
7,0101,01,Central,2025-08,complete,393,15,12,3,10,8


Reconciliation (including 0000): 137243 + 172 = 137415 vs in-scope night 137415
Reconciliation (excluding 0000): 126919 + 166 = 127085 vs 127085


,occ_year_month,in_scope_offenses,night_incl_0000,time_0000,night_excl_0000,calendar_coverage,days_since_month_end_to_last_report
0,2025-01,18295,6539,641,5898,complete,605
1,2025-02,17764,6272,526,5746,complete,577
2,2025-03,19241,6942,544,6398,complete,546
3,2025-04,18441,6684,543,6141,complete,516
4,2025-05,19721,7169,536,6633,complete,485
5,2025-06,18099,6942,539,6403,complete,455
6,2025-07,19245,7206,521,6685,complete,424
7,2025-08,19250,7185,547,6638,complete,393
8,2025-09,18333,6580,487,6093,complete,363
9,2025-10,18888,6772,479,6293,complete,332


,RPT Dist No,occ_year_month,raw_recount_incl_0000,table_incl_0000,raw_recount_excl_0000,table_excl_0000,match
0,1912,2025-06,2,2,1,1,True
1,1566,2025-09,7,7,7,7,True
2,2114,2025-05,4,4,4,4,True


,join,key,left_rows,right_rows,result_rows,extra_rows,right_duplicate_keys,unmatched_left_rows,right_keys_not_in_left
0,district-month night counts <- month context,occ_year_month,21290,21,21290,0,0,0,[]
1,unassigned-district night counts <- month context,occ_year_month,54,21,54,0,0,0,"[2025-04, 2025-05, 2025-06, 2025-07, 2025-08, ..."
2,in-scope month summary <- month context,occ_year_month,21,21,21,0,0,0,[]


,join,key,left_rows,right_rows,result_rows,extra_rows,unmatched_left_rows
0,"rows -> per-CaseNo counts (case_rows_in_file, ...",CaseNo,489955,440121,489955,0,0
1,rows -> district median point (coord_far_from_...,RPT Dist No,489274,1161,489274,0,0


,description,rows,rows_in_scope,distance_missing,far_flag_true
area_only_rpt_dist_no,RPT Dist No holds only the 2-digit area code,558,504,558,0
other_not_mappable,"4-digit district but coordinates missing, (0, ...",123,83,123,0


,rows
looked_up,489274
area_only_not_looked_up,558
other_not_looked_up,123
sum,489955
all_rows,489955


**Result:**
- **21,290** observed district-month rows across 1,140 districts.
- Including 0000: 137,243 district rows + 172 unassigned = **137,415**, which
  equals the in-scope night total.
- Excluding 0000: 126,919 + 166 = **127,085**.
- The independent recount of 3 cells, done with plain text filters on `raw`
  (not with the cleaning code), matches the table.
- September 2026 is kept with its observed counts and labeled `partial`.

The most recent months have had the least time to collect late reports.

**Join audits:**

| Join (key `occ_year_month`, many-to-one) | Left rows | Right rows | Result rows | Unmatched left | Duplicate right keys |
|---|---|---|---|---|---|
| District-month ← month context | 21,290 | 21 | 21,290 | 0 | 0 |
| Unassigned ← month context | 54 | 21 | 54 | 0 | 0 |
| Month summary ← month context | 21 | 21 | 21 | 0 | 0 |

No join added or dropped rows. In the unassigned join, 17 of the 21 months
have no unassigned night records. Those right-side keys are simply unused,
which is expected for a left join. Lookups:
- The `CaseNo` count lookup matched all 489,955 rows, with values equal to an
  independent recount.
- The district-median lookup covers only **eligible rows**: map-eligible rows
  with a 4-digit district. That is 489,274 rows before the lookup and 489,274
  with a distance after it, with **0 unmatched among eligible rows**. All their
  districts (1,161) have street-address reference rows.
- Rows **not looked up** are reported separately:
  - 558 rows with an area-only `RPT Dist No` (504 in scope);
  - 123 rows with a 4-digit district but non-mappable coordinates (83 in scope).

  All 681 keep a **missing** distance and a `False` far-from-district flag.
  `False` here means "not assessed", not "close to the district median".
  No median is filled in.
- Reconciliation: 489,274 + 558 + 123 = 489,955 rows.

### Step 8: Record checks: actual results vs. the expectations above

In [14]:
record_checks = results["record_checks"]
display(record_checks)
print(f"{int(record_checks['match'].sum())} of {len(record_checks)} expected fields match.")

,record,UniqueNIBRNo,field,original,expected,actual,match
0,R1,25088246_290_2,in_scope,Date OCC='2025 May 15 02:21:00 PM',True,True,True
1,R1,25088246_290_2,occ_year_month,Date OCC='2025 May 15 02:21:00 PM',2025-05,2025-05,True
2,R1,25088246_290_2,RPT Dist No,RPT Dist No='0558',0558,0558,True
3,R1,25088246_290_2,AREA NAME,AREA NAME='Harbor ',Harbor,Harbor,True
4,R1,25088246_290_2,TotalOffenseCount,TotalOffenseCount='3',3,3,True
5,R1,25088246_290_2,TotalVictimCount,TotalVictimCount='5',5,5,True
6,R1,25088246_290_2,case_rows_in_file,CaseNo='25088246',3,3,True
7,R1,25088246_290_2,toc_mismatch,TotalOffenseCount='3',False,False,True
8,R1,25088246_290_2,Gang-related Crime,Gang-related Crime='Yes',True,True,True
9,R1,25088246_290_2,map_eligible,LAT='33.746'; LON='-118.2817',True,True,True


53 of 53 expected fields match.


**Result:** all 53 fields across R1–R5 match the expectations written before the
transformations. `outputs/record_checks.md` holds the same comparison, with each
record's original values.

### Step 9: Sample of up to 50 cleaned rows

`cn.select_sample` picks rows deterministically (`seed=576`):
1. the 5 record-check rows and the other 2 rows of case 25088246;
2. up to 2 rows from each difficult-case category, such as 0000 and 1200 times,
   night-window boundaries, 2026-09-19, area-only districts, coordinate flags,
   code and case-total problems, weapon missing/multiple/unknown, and each
   out-of-scope reason;
3. typical in-scope rows with no flags, to fill up to 50.

Each row appears once, under its first matching reason. The full dataset is
still cleaned and validated; the sample only illustrates it.

In [15]:
sample = results["sample"]
print("Sample rows:", len(sample))
display(sample["sample_reason"].value_counts().to_frame("rows"))
display(sample[["sample_order", "sample_reason", "UniqueNIBRNo", "Date OCC", "Time OCC", "RPT Dist No",
                "in_scope", "night", "district_eligible", "map_eligible"]].head(15))

Sample rows: 50


,rows
sample_reason,
typical in-scope row with no flags,7
same case as R1 (multi-offense case),2
Time OCC 0000 (night window),2
RPT Dist No holds only the area code,2
coordinates missing,2
"over 5 km from district median, inside LA box (review flag)",2
LOCATION equals RPT Dist No (suspected generalized),2
NIBR Code missing,2
TotalOffenseCount differs from rows in file,2


,sample_order,sample_reason,UniqueNIBRNo,Date OCC,Time OCC,RPT Dist No,in_scope,night,district_eligible,map_eligible
0,1,record check R1: Multi-offense case (3 rows); ...,25088246_290_2,2025-05-15 14:21:00,1421,0558,True,False,True,True
1,2,record check R2: Time OCC 0000; month 2026-08;...,26172245_220_0,2026-08-31 00:00:00,0000,1555,True,True,True,True
2,3,"record check R3: Area-only RPT Dist No; (0, 0)...",C259005916_23H_0,2025-02-10 15:00:00,1500,10,True,False,False,False
3,4,record check R4: Scope boundary: occurred 2024...,25000271_13B_0,2024-12-31 23:55:00,2355,0647,False,True,True,True
4,5,record check R5: Coordinates ~32 km from distr...,26173656_90D_0,2026-09-03 06:45:00,0645,0138,True,False,True,True
5,6,same case as R1 (multi-offense case),25088246_13A_0,2025-05-15 14:21:00,1421,0558,True,False,True,True
6,7,same case as R1 (multi-offense case),25088246_23F_1,2025-05-15 14:21:00,1421,0558,True,False,True,True
7,8,Time OCC 0000 (night window),26156783_36B_0,2025-01-01 00:00:00,0000,1964,True,True,True,True
8,9,Time OCC 0000 (night window),C269007804_26F_0,2026-02-05 00:00:00,0000,1555,True,True,True,True
9,10,Time OCC 1200 (heaped value),26053014_240_0,2026-02-27 12:00:00,1200,0531,True,False,True,True


### Step 10: Automated validation and output files

Every check compares an actual result with an expectation. The expected values
come from the full-data inspection in plan.md, or are fixed rules such as
"rows unchanged". Failures are listed, not hidden. The output files are then
written to `outputs/`, and the SHA-256 of the original CSV is checked again to
show it was not modified.

In [16]:
validation = pd.DataFrame(results["validation"])
print(f"Validation: {int(validation['passed'].sum())} passed, {int((~validation['passed']).sum())} failed")
display(validation[~validation["passed"]])  # empty table = no failures
display(validation)

OUTPUT_DIR = Path("outputs")
output_hashes = cn.write_outputs(results, OUTPUT_DIR)
display(pd.Series(output_hashes, name="SHA-256").to_frame())

sha_after = cn.sha256_file(file_path)
print("Original CSV unchanged after run:", sha_after == sha_before == cn.EXPECTED_SHA256)

Validation: 104 passed, 0 failed


,check,expected,actual,passed,note


,check,expected,actual,passed,note
0,raw CSV SHA-256 before run,97a8dbadafe60eee4b4d86e99769df3aaa23307d455392...,97a8dbadafe60eee4b4d86e99769df3aaa23307d455392...,True,
1,raw CSV SHA-256 after run (file unchanged),97a8dbadafe60eee4b4d86e99769df3aaa23307d455392...,97a8dbadafe60eee4b4d86e99769df3aaa23307d455392...,True,
2,raw rows,489955,489955,True,
3,raw columns,31,31,True,
4,only empty fields became NA on load,True,True,True,
...,...,...,...,...,...
99,lookup [district median]: looked up + area-onl...,489955,489955,True,
100,sample rows <= 50,True,True,True,
101,sample UniqueNIBRNo unique,True,True,True,
102,sample includes R1-R5,True,True,True,


,SHA-256
cleaning_decision_log.md,a388510bc6bf4decdecec4ad6c3b84fedd94bd75b62130...
district_month_night.csv,a621b862544ff869ec5348cc66870ac867c9a0ccf767e2...
nibrs_clean.csv,226f576c80fc87ed2580a1cf5598fff3bfeea35c5c279e...
record_checks.md,5b75ad7c7377232a7a3b7c4006e264ca9e942c95eb7941...
sample_cleaned_rows.csv,7af0b0dacd3d1f50bd101c7376a54afafeb0dd2f37a027...
validation_summary.json,652097e5ba4f67201a65da3e9ddaddd492870594a80216...


Original CSV unchanged after run: True


### Reproducibility

To confirm the notebook recreates the submitted results:

1. **Restart Kernel → Run All** in VS Code. This rewrites `outputs/`.
2. Run the script into a **separate temporary folder**:
   `python clean_nibrs.py --outdir %TEMP%\nibrs_repro` (Windows `cmd`), or
   `python clean_nibrs.py --outdir $env:TEMP\nibrs_repro` (PowerShell).
3. Compare the SHA-256 hashes the script prints with the table above. They should
   be identical. `outputs/` is never deleted.

### Remaining limitations

- The counts are **reported offenses**, not crime rates or individual risk. There
  is no exposure denominator, and they do not show that lighting causes offenses.
- Nighttime is a fixed-clock approximation (20:00–06:00). `Time OCC` 0000 and
  1200 are heaped, and the source does not explain why.
- Any month can still receive late reports. September 2026 is calendar-partial
  in this snapshot.
- 504 in-scope records have no 4-digit district. 8,217 have a suspected
  generalized location. 478 had (0, 0) coordinates. 1,848 are flagged for
  distance review.
- `TotalOffenseCount` and `TotalVictimCount` are inconsistent in some cases. A
  distinct case is not necessarily a distinct incident. The meaning of the
  `CaseNo` prefixes is undocumented.
- Streetlight assets are context only. They are not measurements of lighting
  quality or outages.
- The download date of the CSV is unconfirmed. The data is identified by its
  SHA-256.

### AI assistance and the suggestion I checked

AI tools used:
- **ChatGPT** (OpenAI), for planning and review: drafting my instructions and
  reviewing the plan and results.
- **Claude Code** (Anthropic), for implementation: inspecting the data, drafting
  `plan.md`, and writing `clean_nibrs.py`, these notebook sections, and the outputs.

Codex was not used. Following the handout's alternative, I checked one cleaning
decision that Claude Code suggested:

> *Rows whose `LOCATION` equals their `RPT Dist No` appear to share one
> district-level coordinate rather than a hundred-block location. Flag them
> `location_suspected_generalized` and keep them in district counts.*

My independent check is in the cell below, and its interpretation follows it.

**Independent Check**

In [17]:
district = "0984"

# Records whose location is recorded as the district number.
generalized = raw.loc[
    (raw["LOCATION"] == district)
    & (raw["RPT Dist No"] == district)
]

print("Matching records:", len(generalized))
print("Distinct coordinate pairs:")
display(generalized[["LAT", "LON"]].drop_duplicates())

# Compare with other location descriptions in the same district.
display(
    raw.loc[
        (raw["RPT Dist No"] == district)
        & (raw["LOCATION"] != district),
        ["LOCATION", "LAT", "LON"]
    ].head(10)
)

Matching records: 5
Distinct coordinate pairs:


,LAT,LON
128,34.149,-118.4537


,LOCATION,LAT,LON
1607,14500 Benefit St,34.1491,-118.4491
1621,14600 Dickens St,34.1504,-118.4518
1695,14500 Benefit St,34.1491,-118.4491
1759,14700 Valley Vista Blvd,34.1458,-118.4556
1760,14700 Valley Vista Blvd,34.1458,-118.4556
2868,14800 Dickens St,34.1509,-118.4561
2910,14600 Dickens St,34.1504,-118.4518
3065,14700 Dickens St,34.1506,-118.454
8755,14800 Dickens St,34.1509,-118.4561
15375,4300 Van Nuys Blvd,34.1491,-118.4491


**Interpretation of my independent check (district 0984)**

What I observed:
- 5 records have `LOCATION` equal to `RPT Dist No` (`0984`).
- All 5 share one coordinate pair: (34.149, -118.4537).
- The street-address records shown for district 0984 contain several
  coordinate pairs, unlike the five matching records, which all share one pair.

This **supports** flagging these records as `location_suspected_generalized`:
in this district, rows whose location is just the district number carry one
shared point rather than their own hundred-block coordinates.

It does **not** prove how LAPD assigned those coordinates; the source does not
document it. I checked one district only (0984). I did not check a second
district and did not inspect the points on a map. The flag remains a review
flag. These records stay in district counts, and the dashboard step decides
how to show them on a map.